# 16 | Nested Training-Size GroupKFold: Random Forest vs Weighted CNN

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Aim:** Test whether the Stage 15 increase from 500 to 1,000 Train windows per person generalises across **three subject-independent development folds**. Each fold uses identical 500-cap Validation rows for all four conditions. Original Test participants S023/S032 remain excluded. This is exploratory development CV.

## 01. Import reusable research code

All model fitting, sample selection and metric calculations remain in documented Python modules.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Find repository and import the existing data and evaluation utilities.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root: raise RuntimeError("Open from within the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, SENSOR_COLUMNS, ACTIVITY_NAMES, prepare_harth_nested_window_cohort
from core.training_scale_cv import evaluate_nested_training_group_cv
from core.group_cross_validation import compare_models_by_subject
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 02. Reconstruct exactly nested source windows

The source-specific adapter constructs chronological, pure-label windows and preserves each original cap-500 window in the cap-1000 expanded dataset. The same two original Test IDs remain excluded.

In [ ]:
# Prepare all 29 development people from cached official HARTH source files.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    base_cap=500, expanded_cap=1000,
    window_size=100, step_size=50, sampling_rate=50,
)
assert set(metadata["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
assert int(metadata["in_base_500"].sum()) == 14201
display(overview)
print("Expanded signal windows:", X.shape)
print("Original base validation candidate windows:", int(metadata["in_base_500"].sum()))

## 03. Audit class coverage before fitting models

All four conditions are evaluated on the same held-out base windows within each fold. Extra windows from held-out participants are never permitted in Train.

In [ ]:
# Compare how original versus added windows contribute to classes.
counts = pd.DataFrame({
    "base": metadata.loc[metadata["in_base_500"], "label"].value_counts(),
    "expanded": metadata["label"].value_counts(),
}).reindex(TASK_CLASSES).fillna(0).astype(int)
counts.index = [ACTIVITY_NAMES[code] for code in TASK_CLASSES]
counts["additional"] = counts["expanded"] - counts["base"]
display(counts)
assert (counts["base"] > 0).all()
assert (counts["additional"] >= 0).all()

## 04. Compare four matched training conditions in three folds

Each of three GroupKFold partitions refits RF 500, RF 1000, Weighted CNN 500 and Weighted CNN 1000. Both CNN configurations use the same 5-epoch training budget and Train-only normalisation/loss weighting. **Six CNN training runs** will be performed; CPU execution may be lengthy.

In [ ]:
# Every model in a fold shares the same held-out base-window IDs.
FOLDS = 3
EPOCHS = 5
fold_results, subject_results, class_results, prediction_records = evaluate_nested_training_group_cv(
    windows=X,
    metadata=metadata,
    classes=TASK_CLASSES,
    channel_names=SENSOR_COLUMNS,
    n_splits=FOLDS,
    epochs=EPOCHS,
    batch_size=128,
    learning_rate=1e-3,
    random_state=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    forbidden_subject_ids=RESERVED_TEST,
)
display(fold_results[[
    "fold", "model", "train_subjects", "validation_subjects",
    "train_windows", "validation_windows", "macro_f1", "balanced_accuracy",
]].round(4))

## 05. Compare fold-wise Macro F1 under fixed Validation

Results from each fold are development diagnostics. Cross-fold averages are descriptive and should not be treated as independent-test estimates.

In [ ]:
# Summarise the same fixed eight-class metric for all four conditions.
summary = fold_results.groupby("model", as_index=False).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_sd=("macro_f1", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
)
display(summary.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
for name, table in fold_results.groupby("model", sort=False):
    ax.plot(table["fold"], table["macro_f1"], marker="o", label=name)
ax.set(title="Three matched group-held-out folds: Training 500 vs 1000",
       xlabel="Fold", ylabel="Macro F1", ylim=(0, 1),
       xticks=list(range(1, FOLDS + 1)))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 06. Inspect class-wise out-of-fold classification

The same original 500-cap Validation windows are scored by all models. Evaluate whether the additional Train windows help Shuffling and stair activities rather than relying only on aggregate results.

In [ ]:
# Report precision, recall and F1 for identical out-of-fold source labels.
class_results["activity"] = class_results["class"].map(
    lambda code: ACTIVITY_NAMES[int(code)]
)
display(class_results[[
    "model", "activity", "support", "precision", "recall", "f1"
]].round(4))

# Plot four F1 bars per activity in a single grouped chart.
fig, ax = plt.subplots(figsize=(14, 5))
names = ["RF 500", "RF 1000", "Weighted CNN 500", "Weighted CNN 1000"]
positions = np.arange(len(TASK_CLASSES))
width = 0.19
for i, name in enumerate(names):
    score = class_results.loc[class_results["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i - 1.5)*width, score, width=width, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
              rotation=35, ha="right")
ax.set(title="Out-of-fold per-class F1", ylabel="F1", ylim=(0, 1))
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 07. Review paired changes for each participant

This analysis uses one fixed-task Macro F1 per held-out development participant. Differences are descriptive rather than statistical significance tests.

In [ ]:
# Compare increasing Train size separately for RF and Weighted CNN.
comparisons = [
    ("RF 1000", "RF 500"),
    ("Weighted CNN 1000", "Weighted CNN 500"),
]
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, (larger, smaller) in zip(axes, comparisons):
    paired, difference = compare_models_by_subject(
        subject_results, model_a=larger, model_b=smaller,
        score_column="macro_f1_fixed_task",
    )
    display(pd.DataFrame([difference]).round(4))
    ax.bar(paired["subject_id"], paired["difference_a_minus_b"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{larger} minus {smaller}",
           ylabel="Per-subject Macro F1 difference")
    ax.grid(axis="y", alpha=0.25)
axes[-1].tick_params(axis="x", rotation=75)
axes[-1].set_xlabel("Held-out development participant")
fig.tight_layout()
plt.show()

## 08. Research quality gate

Confirm every original base source row is held out exactly once per model, that no Test IDs are present and that expanded training never accesses participants in that fold's Validation.

In [ ]:
# Inspect provenance and group isolation after all evaluation runs.
original_rows = set(np.flatnonzero(metadata["in_base_500"].to_numpy()))
assert not (set(subject_results["subject_id"]) & RESERVED_TEST)
assert set(subject_results["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert len(fold_results) == 4 * FOLDS
assert len(subject_results) == 4 * len(DEVELOPMENT_SUBJECTS)
assert len(prediction_records) == 4 * len(original_rows)
for model, table in prediction_records.groupby("model"):
    assert set(table["source_row"]) == original_rows
    assert table["source_row"].is_unique
assert fold_results["macro_f1"].between(0, 1).all()
print("Matched nested training-size grouped CV: PASSED")
print("Original protected Test subjects loaded or scored: NO")
print("Independent full-dataset benchmark: NOT ESTABLISHED")